## MY FIRST MACHINE LEARNING PROJECT

In [1]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import svm

%matplotlib inline

In [2]:
import importlib

import utils

importlib.reload(utils)

from utils import w2d

In [3]:
#  saving the image to a folder

import os
import shutil

img_dirs = []
path_to_img = './cleaned/'
path_to_cropped_images = './cleaned2/'

for entry in os.scandir(path_to_img):

    if entry.is_dir():
        img_dirs.append(entry.path)


In [4]:

if os.path.exists(path_to_cropped_images):
    shutil.rmtree(path_to_cropped_images)
os.makedirs(path_to_cropped_images)

In [5]:
img_dirs

['./cleaned/aaron_taylor_johnson',
 './cleaned/abigail_breslin',
 './cleaned/adam_sandler',
 './cleaned/adrianne_palicki',
 './cleaned/alan_arkin',
 './cleaned/alec_baldwin',
 './cleaned/alexis_thorpe',
 './cleaned/amanda_seyfried',
 './cleaned/amy_adams',
 './cleaned/andrew_garfield',
 './cleaned/angelina jolie',
 './cleaned/angelina_jolie',
 './cleaned/anjelica_huston',
 './cleaned/annasophia_robb',
 './cleaned/anna_kendrick',
 './cleaned/anna_paquin',
 './cleaned/anthony_hopkins',
 './cleaned/babar azam',
 './cleaned/barbra_streisand',
 './cleaned/benedict_cumberbatch',
 './cleaned/ben_affleck',
 './cleaned/ben_kingsley',
 './cleaned/ben_stiller',
 './cleaned/bette_midler',
 './cleaned/betty_white',
 './cleaned/bill_murray',
 './cleaned/brad pitt',
 './cleaned/bradley_cooper',
 './cleaned/brad_pitt',
 './cleaned/brenda_fricker',
 './cleaned/bruce_willis',
 './cleaned/bryan_cranston',
 './cleaned/buster_keaton',
 './cleaned/cameron_diaz',
 './cleaned/carey_mulligan',
 './cleaned/caro

In [6]:
#  begin cropping

# celebrity_filename_list = create_filename_list(img_dirs)

In [7]:

import json

# with open('cleb3.json', 'w') as f:
#     json.dump(celebrity_filename_list, f, indent=4)


with open("cleb.json", "r") as f:
    celebrity_filename_list = json.load(f)

In [8]:
celebrity_filename_list

{'aaron taylor johnson': ['./cleaned/aaron_taylor_johnson\\face_detected_01ae6051.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_19597470.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_3bdf9f44.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_4015533a.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_40396e81.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_40b0e022.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_4b2d66f9.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_4bbd2080.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_58db0e97.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_5991e2bd.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_5c7cb255.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_65255e23.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_6d1b1999.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_7e3b855d.jpg',
  './cleaned/aaron_taylor_johnson\\face_detected_8991f3cb.jpg',
  './cleaned/aar

In [9]:
#  assign each folder a key

class_dict = {}

count = 0
for celebrity_name in celebrity_filename_list.keys(): # noqa: SIM118
    class_dict[celebrity_name] = count
    count = count + 1
class_dict

{'aaron taylor johnson': 0,
 'abigail breslin': 1,
 'adam sandler': 2,
 'adrianne palicki': 3,
 'alan arkin': 4,
 'alec baldwin': 5,
 'alexis thorpe': 6,
 'amanda seyfried': 7,
 'amy adams': 8,
 'andrew garfield': 9,
 'angelina jolie': 10,
 'anjelica huston': 11,
 'annasophia robb': 12,
 'anna kendrick': 13,
 'anna paquin': 14,
 'anthony hopkins': 15,
 'babar azam': 16,
 'barbra streisand': 17,
 'benedict cumberbatch': 18,
 'ben affleck': 19,
 'ben kingsley': 20,
 'ben stiller': 21,
 'bette midler': 22,
 'betty white': 23,
 'bill murray': 24,
 'brad pitt': 25,
 'bradley cooper': 26,
 'brenda fricker': 27,
 'bruce willis': 28,
 'bryan cranston': 29,
 'buster keaton': 30,
 'cameron diaz': 31,
 'carey mulligan': 32,
 'carol burnett': 33,
 'cary grant': 34,
 'cate blanchett': 35,
 'catherine zeta jones': 36,
 'channing tatum': 37,
 'charlie hunnam': 38,
 'charlize theron': 39,
 'cher': 40,
 'chloe grace moretz': 41,
 'chris gayle': 42,
 'christian bale': 43,
 'christina ricci': 44,
 'chris

In [10]:
#  splitting the data into x and y

x = []
y = []

for celebrity_name, training_files in celebrity_filename_list.items():
    for training_image in training_files:
        img = cv2.imread(training_image)
        if img is None:
            continue
        scaled_raw_img = cv2.resize(img, (32, 32))
        img_harr = w2d(img, 'db1', 5)
        scaled_img_harr = cv2.resize(img_harr, (32, 32))
        combined_img = np.vstack((scaled_raw_img.reshape(32*32*3, 1), scaled_img_harr.reshape(32*32, 1)))
        x.append(combined_img)
        y.append(celebrity_name)

x = np.array(x).reshape(len(x), -1).astype(float)

In [11]:
x.shape

(5830, 4096)

In [12]:
x[0]

array([104., 123., 168., ..., 211., 239., 220.], shape=(4096,))

In [13]:
#  test training

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=0)

pipe = Pipeline([
    (
        'scaler', StandardScaler(),
    ),
    (
        'svc', SVC(kernel='rbf', C=10)
    )
])
pipe.fit(x_train, y_train)
pipe.score(x_test, y_test)

0.5089163237311386

In [14]:
#  make a model param

model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto', probability=True),
        'params': {
            'svc__C': [1,10,100],
            'svc__kernel': ['rbf', 'linear']
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'randomforestclassifier__n__estimators': [1,5,10]
        }
    },
    'logistic_regression': {
        'model': LogisticRegression(solver='lbfgs'),
        'params': {
            'logisticregression__C': [1,5,10]
        }
    }
}

In [ ]:
# determine the best model
import pandas as pd

scores = []
best_estimators = {}

for algo, mp in model_params.items():
    pipe = make_pipeline(StandardScaler(), mp['model'])
    clf = GridSearchCV(pipe, mp['params'], cv=5, return_train_score=False)
    clf.fit(x_train, y_train)
    scores.append({
        'model': algo,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

    best_estimators[algo] = clf.best_estimator_

df = pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])
df

c:\Users\UZER\Documents\AI TUT\projects\img-classifier-pro\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\UZER\Documents\AI TUT\projects\img-classifier-pro\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
